# Chapter 17: Using Graphics Hardware

    Source orientation: printed pages 461-502; physical PDF pages 478-519. This notebook is standalone: it uses the source span only to orient the topics and then teaches the material through original explanations, generated visuals, executable experiments, and reproducible checks.

    ## Chapter Question

    How can we make using graphics hardware inspectable instead of merely descriptive? In this course the answer is to translate each idea into a small computational object: an array, a vector, a transform, a field, a sampling rule, a diagram, or a measured invariant. For this chapter the central vocabulary is GPU pipeline, buffers, state, shaders, attributes, and textures. The notebook is written for a reader who wants to understand the graphics idea well enough to test it, adapt it, and recognize when an implementation has gone wrong.

    ## Why This Chapter Matters

    Computer graphics is unusually visual even when the underlying topic looks algebraic or architectural. A renderer can fail because a sign convention is wrong, because a sample distribution is biased, because a color value is encoded in the wrong space, because a mesh adjacency relation is inconsistent, or because a transform has been applied in the wrong order. The best way to learn the chapter is therefore to build small objects whose behavior can be seen and audited. The figures below are not illustrations pasted next to the text; they are generated evidence for the claims the notebook makes.

    This chapter also acts as a bridge to the rest of the book. The ideas here feed later notebooks whenever pixels must be generated, geometry must be moved, light must be evaluated, a signal must be sampled, or a human viewer must interpret the result. The examples are deliberately compact, because compact examples make invariants easier to inspect. A production renderer or visualization system may be much larger, but the same checks still apply: dimensions must agree, ranges must be controlled, normals must be normalized, samples must match their intended measure, and every visual output must have a reason to exist.

    ## Translation guide

    | Book concept | Computational translation |
    | --- | --- |
    | GPU pipeline | Treat it as a quantity, transformation, image, or data structure that can be generated, measured, and checked in code. |
| buffers | Treat it as a quantity, transformation, image, or data structure that can be generated, measured, and checked in code. |
| state | Treat it as a quantity, transformation, image, or data structure that can be generated, measured, and checked in code. |
| shaders | Treat it as a quantity, transformation, image, or data structure that can be generated, measured, and checked in code. |
| attributes | Treat it as a quantity, transformation, image, or data structure that can be generated, measured, and checked in code. |
| textures | Treat it as a quantity, transformation, image, or data structure that can be generated, measured, and checked in code. |

    The translation guide is the working contract for the notebook. If a paragraph introduces a geometric or imaging claim, a nearby cell either visualizes the claim or records a check that would catch a common mistake. This does not mean every idea needs a large figure. Sometimes a small numeric assertion is better than a plot. Sometimes a plotted construction is more useful than a long derivation. The point is to keep the explanation inspectable.

    ## Route Through The Chapter

    1. Establish the local vocabulary and the chapter question.
    2. Build a generated visual artifact that exposes the chapter's dependency structure.
    3. Build a second visual artifact focused on the most practical computation in the chapter.
    4. Record numeric checks that can be re-run after edits.
    5. Finish with an applied lab and takeaways that connect the chapter to later rendering, modeling, image, or visualization work.

    ## Visual storyboard

    The planned visual sequence is: a mini GPU style raster artifact; buffer and shader-stage reasoning; depth/interpolation checks. Each artifact is saved under `artifacts/chapter-17/` so the notebook can be executed from its own folder while still keeping outputs in the book-local artifact tree. The first figure is a concept dependency map. It helps the reader see which ideas should be held together when debugging or extending an implementation. The second figure is chapter-specific and turns one of the main algorithms or representations into an inspectable object.

    ## Concept Notes

    The first habit this chapter builds is separating representation from interpretation. A tuple of numbers may be a color, a point, a direction, a homogeneous coordinate, a probability sample, or a display-space pixel. The operations that are valid for one interpretation can be invalid for another. The notebook therefore names the meaning of every important array and records the checks that preserve that meaning. This is especially useful in computer graphics because the final output is often an image that can look plausible even when one part of the computation is conceptually wrong.

    The second habit is to use deliberately small scenes and synthetic data. Synthetic examples are not a compromise here; they are a microscope. A controlled triangle, sphere, color ramp, signal, curve, mesh, or graph lets us know the expected answer before the system becomes visually complicated. Once the simple case behaves correctly, the same machinery can be reused with richer assets. This is also why the notebook avoids external image or model files. All artifacts are reproducible from code cells.

    The third habit is to pair visuals with invariants. A plot can reveal structure quickly, but a check can keep the structure from silently drifting when code changes. In this chapter the checks emphasize quantities such as sums, ranges, reconstruction errors, normalization, monotonicity, topology counts, sample statistics, or transform consistency. These checks are intentionally small enough to read. They are not a substitute for a full test suite, but they make the notebook robust as a teaching artifact.

    ## Worked Example

    The worked example below uses the shared course helpers rather than a hidden external framework. The goal is not to build a production graphics engine inside a notebook. The goal is to make the core idea of using graphics hardware concrete enough that a learner can inspect inputs, outputs, and failure modes. The generated visual is saved as a durable PNG, displayed inline, and summarized by a JSON file containing the numeric checks.

    ## Applied lab

    Modify one parameter in the chapter-specific visual and predict which invariant should change. For example, change a vector, a light direction, a sampling seed, a transform, a tone curve exposure, a control point, or a graph layout seed. Re-run the visual and compare the JSON checks before and after the change. A good lab response names both a visual change and a numeric change. If the visual changes but the numeric check does not, the check is probably too weak. If the numeric check changes but the visual does not, the display may be hiding the behavior that matters.

    A second lab is to design a failure case. Choose one assumption from the translation guide and violate it: pass an unnormalized normal, move a point outside a triangle, set a field threshold too high, use an undersampled signal, apply a transform in the wrong order, or clip a color value too early. The purpose is to learn what the failure looks like. In graphics, recognizing the visual signature of an error is often faster than single-stepping through thousands of repeated pixel or vertex operations.

    ## Sanity checks

    The final cells assert that generated artifacts exist, are nonblank, and have nontrivial file size. They also write the chapter's numeric check summary. These checks make the notebook safe to execute with `nbclient` and useful for later QC.

    ## Takeaways

    - Using Graphics Hardware becomes easier when concepts are converted into arrays, functions, diagrams, and invariants.
    - Visuals should expose structure or failure modes, not decorate the page.
    - The artifact JSON file is part of the lesson because it records what the figure is supposed to prove.
    - The small example is a seed for larger graphics systems: keep the checks, then scale the data.

In [ ]:
CHAPTER = 17
TITLE = 'Using Graphics Hardware'
CONCEPTS = ['GPU pipeline', 'buffers', 'state', 'shaders', 'attributes', 'textures']
PRINTED_PAGES = '461-502'
PDF_PAGES = '478-519'

In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the FCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / f"chapter-{CHAPTER:02d}"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
from utils.artifacts import assert_artifacts, display_artifact, save_json
from utils.course_visuals import create_chapter_visuals
from utils.notebook_checks import assert_nonblank_image

In [ ]:
result = create_chapter_visuals(CHAPTER, TITLE, CONCEPTS, f"chapter-{CHAPTER:02d}")
artifact_paths = result["paths"]
check_path = result["check_path"]
result["checks"]

In [ ]:
for path in artifact_paths:
    display_artifact(path, width=760)
display_artifact(check_path)

In [ ]:
artifact_records = assert_artifacts([*artifact_paths, check_path])
image_records = [assert_nonblank_image(path) for path in artifact_paths]
assert result["checks"]["concept_count"] == len(CONCEPTS)
artifact_records

In [ ]:
final_report = {
    "chapter": CHAPTER,
    "title": TITLE,
    "printed_pages": PRINTED_PAGES,
    "pdf_pages": PDF_PAGES,
    "artifact_count": len(artifact_paths),
    "nonblank_images": len(image_records),
    "checks": result["checks"],
}
final_path = save_json(final_report, f"chapter-{CHAPTER:02d}", "final-sanity.json")
display_artifact(final_path)
final_report